# Finlora Fintech — Fraud Detection & Risk Scoring
## Notebook 01: Data Loading, Profiling & Cleaning

**Project goal.** Finlora runs a 24/7 digital financial platform. Today it catches
fraud with **fixed, rule-based thresholds** on amount, velocity and geography. As
volume grows, these blunt rules miss subtle fraud *and* over-flag legitimate
customers. The project replaces them with an **ML risk-scoring model** that assigns
every transaction a continuous fraud probability.

**This notebook's job (Project Task)**
1. Load the **transactions** table (`tx`) and **accounts** table (`acc`).
2. Profile shape, dtypes, nulls, duplicates, referential integrity and class balance.
3. Clean both tables (missing values, standardize timestamps, validate categories,
   treat device nulls, handle refunds/outliers, confirm `is_cross_border`).
4. **Merge** the two tables into a single dataframe **`df`**.
5. Save the cleaned, merged dataset as **`test_cleaned_data.csv`** in `processed_data/`.
6. Write a short **cleaning log**.

> Why a separate cleaning notebook? Cleaning decisions are documented here once and
> reused by every later notebook. The saved CSV is the single source of truth.

In [9]:
# --- Imports ---------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings, json, pickle, os
warnings.filterwarnings("ignore")

### 1. Load the raw datasets

We load:
- **`tx`** — transactions (expected ~126,000 rows, 24 columns)
- **`acc`** — accounts (expected ~7,200 rows, 8 columns)

These two variable names are used throughout the project so the cleaning steps
map cleanly to the task description.

In [10]:
# Load the two source tables. 
tx = pd.read_csv(r"C:\Users\super\Documents\GitHub\finlora_fintech\data\raw_data\finlora_transactions.csv")
acc = pd.read_csv(r"C:\Users\super\Documents\GitHub\finlora_fintech\data\raw_data\finlora_accounts.csv")

print("Transactions (tx):", tx.shape)
print("Accounts    (acc):", acc.shape)


Transactions (tx): (126000, 24)
Accounts    (acc): (7200, 8)


In [11]:
tx.head()

,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,...,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,...,9143.33,0,NG,NG,0,NaN,NaN,903,Completed,0
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,...,9143.33,0,NG,NG,0,NaN,NaN,969,Declined,0
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,...,9143.33,0,NG,NG,0,DEV-9055235108,0.0,970,Completed,0
3,FLR250428101772,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-28 14:41:59,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,...,14891.16,0,NG,NG,0,NaN,NaN,974,Completed,0
4,FLR250628102827,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-06-28 16:15:47,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,...,49609.59,0,NG,NG,0,DEV-9055235108,0.0,1035,Declined,0


### 2. First look: shape, dtypes, column names

We compare the loaded columns against the data dictionary to make sure nothing
was silently renamed or dropped during the CSV read.

In [12]:
print("=== tx columns ===")
print(tx.dtypes)
print("\n=== acc columns ===")
print(acc.dtypes)

=== tx columns ===
transaction_id                 object
account_id                     object
account_type                   object
kyc_tier                       object
timestamp                      object
day_of_week                    object
hour_of_day                     int64
description                    object
merchant_name                  object
merchant_category              object
channel                        object
amount                        float64
currency                       object
amount_to_avg_ratio           float64
avg_transaction_amount_30d    float64
transaction_velocity_1h         int64
transaction_country            object
home_country                   object
is_cross_border                 int64
device_id                      object
is_new_device                 float64
account_age_days                int64
status                         object
is_fraud                        int64
dtype: object

=== acc columns ===
account_id                      ob

In [33]:
tx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 27 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   transaction_id              126000 non-null  object        
 1   account_id                  126000 non-null  object        
 2   account_type                126000 non-null  object        
 3   kyc_tier                    126000 non-null  object        
 4   timestamp                   126000 non-null  datetime64[ns]
 5   day_of_week                 126000 non-null  object        
 6   hour_of_day                 126000 non-null  int32         
 7   description                 126000 non-null  object        
 8   merchant_name               126000 non-null  object        
 9   merchant_category           126000 non-null  object        
 10  channel                     126000 non-null  object        
 11  amount                      126000 non-

In [34]:
tx.describe()

,timestamp,hour_of_day,amount,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,is_cross_border,is_new_device,account_age_days,is_fraud,device_id_missing,is_new_device_missing,is_refund
count,126000,126000.000000,1.260000e+05,126000.000000,1.260000e+05,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000
mean,2025-11-17 18:17:27.700087296,13.542111,7.964918e+05,7.921400,7.675995e+05,0.011429,0.020198,-0.082849,354.721992,0.026603,0.113762,0.113762,0.002310
min,2025-02-01 00:05:31,0.000000,-1.834602e+06,0.000000,-1.669153e+06,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025-07-11 09:22:28,9.000000,6.310500e+01,0.250000,9.242750e+01,0.000000,0.000000,0.000000,35.000000,0.000000,0.000000,0.000000,0.000000
50%,2025-11-27 22:33:02.500000,14.000000,5.942240e+03,0.980000,5.942240e+03,0.000000,0.000000,0.000000,289.000000,0.000000,0.000000,0.000000,0.000000
75%,2026-03-24 12:34:02.750000128,18.000000,1.208470e+05,1.830000,1.575701e+05,0.000000,0.000000,0.000000,577.000000,0.000000,0.000000,0.000000,0.000000
max,2026-08-30 23:33:21,23.000000,2.691614e+08,31762.690000,1.733621e+08,3.000000,1.000000,1.000000,1682.000000,1.000000,1.000000,1.000000,1.000000
std,NaN,5.260369,4.054387e+06,153.566706,3.290377e+06,0.110323,0.140679,0.371230,331.327033,0.160921,0.317523,0.317523,0.048002


In [35]:
acc.head()

,account_id,account_holder_name,account_type,home_country,currency,kyc_tier,account_created_date,personal_spend_baseline_usd
0,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23
1,FLR-ACC-100001,Bluewave Foods Ltd,Business,NG,NGN,Tier1_Basic,2026-04-27,810.68
2,FLR-ACC-100002,Global Trading LLC,Business,US,USD,Tier2_Verified,2026-03-29,433.67
3,FLR-ACC-100003,Femi Adeyemi,Individual,NG,NGN,Tier3_Enhanced,2025-09-15,19.18
4,FLR-ACC-100004,Ngozi Brown,Individual,NG,NGN,Tier3_Enhanced,2024-06-18,28.68


In [36]:
acc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   account_id                   7200 non-null   object        
 1   account_holder_name          7200 non-null   object        
 2   account_type                 7200 non-null   object        
 3   home_country                 7200 non-null   object        
 4   currency                     7200 non-null   object        
 5   kyc_tier                     7200 non-null   object        
 6   account_created_date         7200 non-null   datetime64[ns]
 7   personal_spend_baseline_usd  7200 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 450.1+ KB


In [37]:
acc.describe()

,account_created_date,personal_spend_baseline_usd
count,7200,7200.000000
mean,2024-12-31 06:32:36,268.854419
min,2021-12-25 00:00:00,3.770000
25%,2024-05-05 18:00:00,43.787500
50%,2025-02-14 00:00:00,87.665000
75%,2025-10-07 00:00:00,286.807500
max,2026-08-29 00:00:00,10469.650000
std,NaN,495.820753


### 3. Data profiling

Before changing anything we record the **state of the raw data**. This profiling
step answers the task's "Data loading and Profiling" requirements:
- null counts per column,
- duplicate counts,
- referential integrity (every `account_id` in `tx` exists in `acc`),
- class balance of `is_fraud`,
- quick distribution summaries.

We keep this as a function so we can call it again later to compare "before vs after".

In [ ]:
git revert HEAD
git push

In [13]:
def profile(df, name):
    """Print a compact profiling report for a dataframe."""
    print(f"\n{'='*70}\nPROFILE: {name}  shape={df.shape}\n{'='*70}")
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    print("\n-- Null counts (cols with nulls only) --")
    if len(nulls):
        print(pd.DataFrame({"nulls": nulls,
                            "pct": (nulls/len(df)*100).round(2)}))
    else:
        print("No nulls.")
    print("\n-- Duplicates --")
    print(f"Full-row duplicates : {df.duplicated().sum()}")
    print(f"Row count           : {len(df)}")
    print("\n-- Numeric summary --")
    num = df.select_dtypes(include=[np.number])
    if len(num.columns):
        print(num.describe().T.round(2))
    return None

profile(tx, "transactions")
profile(acc, "accounts")


PROFILE: transactions  shape=(126000, 24)

-- Null counts (cols with nulls only) --
               nulls    pct
merchant_name  28610  22.71
device_id      14334  11.38
is_new_device  14334  11.38

-- Duplicates --
Full-row duplicates : 0
Row count           : 126000

-- Numeric summary --
                               count       mean         std         min  \
hour_of_day                 126000.0      13.54        5.26        0.00   
amount                      126000.0  796491.82  4054387.23 -1834602.45   
amount_to_avg_ratio         126000.0       7.92      153.57        0.00   
avg_transaction_amount_30d  126000.0  767599.46  3290377.16 -1669153.21   
transaction_velocity_1h     126000.0       0.01        0.11        0.00   
is_cross_border             126000.0       0.02        0.14        0.00   
is_new_device               111666.0       0.03        0.18        0.00   
account_age_days            126000.0     354.72      331.33        0.00   
is_fraud                    126000

In [14]:
# Referential integrity: every account_id used in transactions must exist in accounts.
tx_ids   = set(tx["account_id"].unique())
acc_ids  = set(acc["account_id"].unique())
missing_in_acc = tx_ids - acc_ids          # tx rows that would NOT match acc
orphan_acc     = acc_ids - tx_ids          # accounts with no transactions at all

print("Distinct account_ids in tx       :", len(tx_ids))
print("Distinct account_ids in acc      :", len(acc_ids))
print("tx ids NOT found in acc          :", len(missing_in_acc), "(should be 0)")
print("accounts with no transactions    :", len(orphan_acc), "(ok, just unused accounts)")

Distinct account_ids in tx       : 7144
Distinct account_ids in acc      : 7200
tx ids NOT found in acc          : 0 (should be 0)
accounts with no transactions    : 56 (ok, just unused accounts)


In [15]:
# Class balance of the target. Fraud is rare -> heavily imbalanced.
counts = tx["is_fraud"].value_counts().sort_index()
fraud_rate = tx["is_fraud"].mean()
print("is_fraud counts:")
print(counts)
print(f"\nFraud rate: {fraud_rate*100:.3f}%   (legit: {(1-fraud_rate)*100:.3f}%)")
print("Implication: we must use stratified splitting + class weighting later.")

is_fraud counts:
is_fraud
0    122648
1      3352
Name: count, dtype: int64

Fraud rate: 2.660%   (legit: 97.340%)
Implication: we must use stratified splitting + class weighting later.


### 4. Data cleaning — transactions (`tx`)

We now fix the issues found in profiling. Every decision is written as a short
comment so an entry-level teammate can follow the reasoning.

**Cleaning checklist (from the project task):**
1. Check `transaction_id` uniqueness, remove/flag duplicates.
2. Standardize `timestamp`; re-derive `day_of_week` and `hour_of_day` to confirm.
3. Validate categorical columns for typos / unexpected values.
4. Treat null `device_id` and `is_new_device` (explicit "unknown" + missing-flag).
5. Treat null `merchant_name` as valid for peer transfers and payroll.
6. Review negative amounts (refunds/credits) and extreme outliers.
7. Confirm `is_cross_border` matches `transaction_country` vs `home_country`.

#### 4.1 Duplicate transaction_ids

In [16]:
# Duplicate check: transaction_id should be the unique key for the table.
dup_ids = tx["transaction_id"].duplicated().sum()
dup_rows = tx.duplicated().sum()
print(f"Duplicate transaction_ids : {dup_ids}")
print(f"Full-row duplicates       : {dup_rows}")

if dup_ids:
    tx = tx.drop_duplicates(subset="transaction_id", keep="first")
    print("Dropped duplicate transaction_id rows. New shape:", tx.shape)
else:
    print("No duplicate transaction_ids -> nothing to drop.")

Duplicate transaction_ids : 0
Full-row duplicates       : 0
No duplicate transaction_ids -> nothing to drop.


#### 4.2 Standardize timestamp and re-derive day/hour

In [17]:
# Parse timestamp to a real datetime. Then re-derive day_of_week and hour_of_day
# FROM the timestamp and compare to the supplied values. If they all match, the
# supplied fields are trustworthy; if not, we overwrite with the derived version.
tx["timestamp"] = pd.to_datetime(tx["timestamp"], errors="coerce")

derived_dow  = tx["timestamp"].dt.day_name()
derived_hour = tx["timestamp"].dt.hour

dow_mismatch  = (derived_dow.astype(str)  != tx["day_of_week"].astype(str)).sum()
hour_mismatch = (derived_hour != tx["hour_of_day"]).sum()

print(f"day_of_week mismatches  : {dow_mismatch}")
print(f"hour_of_day mismatches  : {hour_mismatch}")

# Even though they match, we trust the derived values (single source of truth).
tx["day_of_week"]  = derived_dow
tx["hour_of_day"]  = derived_hour
print("Timestamp parsed; day_of_week and hour_of_day confirmed from timestamp.")
print("Timestamp range:", tx["timestamp"].min(), "->", tx["timestamp"].max())

day_of_week mismatches  : 0
hour_of_day mismatches  : 0
Timestamp parsed; day_of_week and hour_of_day confirmed from timestamp.
Timestamp range: 2025-02-01 00:05:31 -> 2026-08-30 23:33:21


#### 4.3 Validate categorical columns

In [18]:
# Validate the categorical columns that the task lists.
cat_cols = ["account_type", "kyc_tier", "channel", "status",
            "currency", "merchant_category", "transaction_country",
            "home_country", "day_of_week"]

for c in cat_cols:
    vals = tx[c].dropna().unique()
    print(f"\n{c}  ({len(vals)} unique)")
    print(sorted(vals.tolist()))


account_type  (2 unique)
['Business', 'Individual']

kyc_tier  (3 unique)
['Tier1_Basic', 'Tier2_Verified', 'Tier3_Enhanced']

channel  (6 unique)
['API/Integration', 'Card Not Present', 'Card Present', 'Mobile App', 'USSD', 'Web Dashboard']

status  (3 unique)
['Completed', 'Declined', 'Reversed']

currency  (4 unique)
['EUR', 'GBP', 'NGN', 'USD']

merchant_category  (15 unique)
['ATM Withdrawal', 'Crypto Exchange', 'Electronics', 'Gambling/Gaming', 'Groceries', 'Healthcare', 'Insurance', 'P2P Transfer', 'Payroll Transfer', 'Restaurants', 'Retail', 'Subscription/SaaS', 'Travel', 'Utilities', 'Wire Transfer']

transaction_country  (5 unique)
['DE', 'FR', 'GB', 'NG', 'US']

home_country  (5 unique)
['DE', 'FR', 'GB', 'NG', 'US']

day_of_week  (7 unique)
['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']


In [19]:
# No typos or unexpected categories were found in the profiling step
# (e.g. account_type is exactly {Individual, Business}, kyc_tier has the 3 tiers,
# channel has the 6 expected channels). We strip whitespace defensively so a
# stray " Individual" would not become a new category.
for c in cat_cols:
    if tx[c].dtype == object:
        tx[c] = tx[c].astype(str).str.strip()
print("Categorical columns trimmed of stray whitespace. Values are clean.")

Categorical columns trimmed of stray whitespace. Values are clean.


#### 4.4 Treat null `device_id` and `is_new_device`

In [20]:
# device_id is null for ~11% of rows. This is NOT bad data: some channels
# (Card Present, USSD) or known-device flows simply do not carry a device id.
# Strategy from the task: an explicit "unknown" category PLUS a missing-flag.
# The flag lets the model learn "missing device" as its own signal.

tx["device_id_missing"] = tx["device_id"].isna().astype(int)
tx["device_id"] = tx["device_id"].fillna("unknown")

# is_new_device is null in exactly the same rows. We treat null as "unknown"
# by introducing -1, and keep a separate missing flag for safety.
tx["is_new_device_missing"] = tx["is_new_device"].isna().astype(int)
tx["is_new_device"] = tx["is_new_device"].fillna(-1).astype(int)

print("device_id nulls after fill :", tx["device_id"].isna().sum())
print("is_new_device values       :")
print(tx["is_new_device"].value_counts(dropna=False))
print("\ndevice_id_missing flag:")
print(tx["device_id_missing"].value_counts())

device_id nulls after fill : 0
is_new_device values       :
is_new_device
 0    107771
-1     14334
 1      3895
Name: count, dtype: int64

device_id_missing flag:
device_id_missing
0    111666
1     14334
Name: count, dtype: int64


#### 4.5 Treat null `merchant_name`

In [21]:
# merchant_name is null for ~23% of rows. For Payroll Transfer and P2P Transfer
# this is EXPECTED: there is no merchant (the counter-party is an account).
# For other categories the nulls are a small number of genuine gaps.

print("merchant_name nulls by category (top):")
print(tx[tx["merchant_name"].isna()]["merchant_category"].value_counts().head())

# We fill all nulls with "unknown". merchant_name is high-cardinality free text
# and will NOT be used directly as a model feature (see feature engineering),
# so a single placeholder is enough and keeps the column tidy.
tx["merchant_name"] = tx["merchant_name"].fillna("unknown")
print("\nmerchant_name nulls after fill:", tx["merchant_name"].isna().sum())

merchant_name nulls by category (top):
merchant_category
Payroll Transfer     17117
P2P Transfer         10715
Groceries              112
Retail                  98
Subscription/SaaS       97
Name: count, dtype: int64

merchant_name nulls after fill: 0


#### 4.6 Negative amounts and extreme outliers

In [22]:
# Negative amounts are refunds/credits and are legitimate. We keep them but
# add an `is_refund` flag so the model can learn the refund pattern explicitly.
neg_count = (tx["amount"] < 0).sum()
print(f"Negative amounts (refunds/credits): {neg_count}")
tx["is_refund"] = (tx["amount"] < 0).astype(int)

# Extreme high amounts: financial data has long tails (large wire transfers,
# payroll batches). We do NOT drop them — they are real and fraud-relevant.
# We WILL use a log transform in feature engineering to tame the skew.
print("\nAmount distribution:")
print(tx["amount"].describe().round(2))
print("\nTop 3 amounts :", tx["amount"].nlargest(3).round(2).tolist())
print("Bottom 3      :", tx["amount"].nsmallest(3).round(2).tolist())

Negative amounts (refunds/credits): 291

Amount distribution:
count    1.260000e+05
mean     7.964918e+05
std      4.054387e+06
min     -1.834602e+06
25%      6.310000e+01
50%      5.942240e+03
75%      1.208470e+05
max      2.691614e+08
Name: amount, dtype: float64

Top 3 amounts : [269161408.78, 269098396.56, 209067896.22]
Bottom 3      : [-1834602.45, -1669153.21, -1301224.99]


#### 4.7 Confirm `is_cross_border` consistency

In [23]:
# is_cross_border must equal 1 exactly when transaction_country != home_country.
expected_cb = (tx["transaction_country"].astype(str) != tx["home_country"].astype(str)).astype(int)
mismatch = (expected_cb != tx["is_cross_border"]).sum()
print(f"is_cross_border mismatches: {mismatch}  (0 means the field is consistent)")

if mismatch:
    tx["is_cross_border"] = expected_cb
    print("Overwrote is_cross_border with the country-based derivation.")
else:
    print("is_cross_border is consistent with country fields -> no change.")

is_cross_border mismatches: 0  (0 means the field is consistent)
is_cross_border is consistent with country fields -> no change.


#### 4.8 Final check on `tx`

In [24]:
print("tx shape after cleaning:", tx.shape)
print("\nRemaining nulls in tx:")
print(tx.isna().sum()[tx.isna().sum() > 0])

helper_columns = [
      "device_id_missing",
      "is_new_device_missing",
      "is_refund",
]

print("\nHelper columns added:", helper_columns)

tx shape after cleaning: (126000, 27)

Remaining nulls in tx:
Series([], dtype: int64)

Helper columns added: ['device_id_missing', 'is_new_device_missing', 'is_refund']


In [25]:
tx.to_csv(r"C:\Users\super\Documents\GitHub\finlora_fintech\data\processed_data\tx_cleaned_data.csv", index=False)

### 5. Data cleaning — accounts (`acc`)

The accounts table was already very clean (no nulls, no duplicates, unique
`account_id`). We still run the same checks for completeness, standardize the
date column, and trim whitespace defensively.

In [26]:
# Duplicates and uniqueness
print("Duplicate account_ids:", acc["account_id"].duplicated().sum())
print("Duplicate full rows  :", acc.duplicated().sum())

# Standardize the created-date column to a real datetime.
acc["account_created_date"] = pd.to_datetime(acc["account_created_date"], errors="coerce")
print("account_created_date range:", acc["account_created_date"].min(),
      "->", acc["account_created_date"].max())

# Trim whitespace on categorical columns.
for c in ["account_type", "home_country", "currency", "kyc_tier"]:
    acc[c] = acc[c].astype(str).str.strip()

print("\nacc nulls after cleaning:")
print(acc.isna().sum()[acc.isna().sum() > 0])
print("(empty above means no nulls remain)")
print("\nacc shape:", acc.shape)

Duplicate account_ids: 0
Duplicate full rows  : 0
account_created_date range: 2021-12-25 00:00:00 -> 2026-08-29 00:00:00

acc nulls after cleaning:
Series([], dtype: int64)
(empty above means no nulls remain)

acc shape: (7200, 8)


In [27]:
acc.to_csv('../data/processed_data/acc_cleaned_data.csv', index=False)

### 6. Merge `tx` and `acc` into `df`

We merge so every transaction is enriched with its **account-level** information
(`personal_spend_baseline_usd`, `account_created_date`, `account_holder_name`).

**Join type:** `left` join from `tx` to `acc` on `account_id`.
- We keep all transactions (the unit we model) and bring in account fields.
- Profiling confirmed every `account_id` in `tx` exists in `acc`, so no rows are lost.

**Duplicate columns:** `account_type`, `home_country`, `currency`, `kyc_tier`
appear in BOTH tables. We drop the copies from `acc` before merging (the
transaction-level copy is authoritative and was already validated).

In [28]:
# Drop the columns acc shares with tx so the merge does not create _x/_y pairs.
acc_for_merge = acc.drop(columns=["account_type", "home_country", "currency", "kyc_tier"])

# Left join: keep every transaction, attach its account info.
df = tx.merge(acc_for_merge, on="account_id", how="left", validate="m:1")
print("Merged df shape:", df.shape)
print("Rows with missing account info:", df["account_holder_name"].isna().sum())
df.head()

Merged df shape: (126000, 30)
Rows with missing account info: 0


,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,...,is_new_device,account_age_days,status,is_fraud,device_id_missing,is_new_device_missing,is_refund,account_holder_name,account_created_date,personal_spend_baseline_usd
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,...,-1,903,Completed,0,1,1,0,Amaka Okafor,2022-08-28,63.23
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,...,-1,969,Declined,0,1,1,0,Amaka Okafor,2022-08-28,63.23
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,...,0,970,Completed,0,0,0,0,Amaka Okafor,2022-08-28,63.23
3,FLR250428101772,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-28 14:41:59,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,...,-1,974,Completed,0,1,1,0,Amaka Okafor,2022-08-28,63.23
4,FLR250628102827,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-06-28 16:15:47,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,...,0,1035,Declined,0,0,0,0,Amaka Okafor,2022-08-28,63.23


### 7. Final profile of the merged `df`

In [29]:
profile(df, "merged df")


PROFILE: merged df  shape=(126000, 30)

-- Null counts (cols with nulls only) --
No nulls.

-- Duplicates --
Full-row duplicates : 0
Row count           : 126000

-- Numeric summary --
                                count       mean         std         min  \
hour_of_day                  126000.0      13.54        5.26        0.00   
amount                       126000.0  796491.82  4054387.23 -1834602.45   
amount_to_avg_ratio          126000.0       7.92      153.57        0.00   
avg_transaction_amount_30d   126000.0  767599.46  3290377.16 -1669153.21   
transaction_velocity_1h      126000.0       0.01        0.11        0.00   
is_cross_border              126000.0       0.02        0.14        0.00   
is_new_device                126000.0      -0.08        0.37       -1.00   
account_age_days             126000.0     354.72      331.33        0.00   
is_fraud                     126000.0       0.03        0.16        0.00   
device_id_missing            126000.0       0.11      

### 8. Save the cleaned dataset

We save the cleaned, merged dataframe as **`test_cleaned_data.csv`** inside the
`processed_data/` folder. This is the single analysis-ready dataset used by every
later notebook (EDA, feature engineering, modeling).

In [30]:
df.to_csv('../data/processed_data/test_cleaned_data.csv', index=False)

### 9. Cleaning log

A short, human-readable record of every cleaning decision and its rationale.
This is saved next to the data so reviewers can audit the pipeline.

### Cleaning Log

**"transactions":** "finlora_transactions.csv",
    
  **"accounts":** "finlora_accounts.csv"
  
  **"transactions_raw_rows":** 126000
  
  **"accounts_raw_rows":** 7200
  

  **"duplicate_transaction_ids_dropped":** 0

  **"duplicate_full_rows_dropped":** 0

  **"timestamp_parsing":** "Converted to datetime; day_of_week and hour_of_day re-derived from timestamp and confirmed to match the supplied values (0 mismatches)."

  **"categorical_validation":** "All categorical columns validated; no typos found; whitespace stripped defensively."

  **"device_id_nulls_pct_before":** 11.38

  **"device_id_strategy":** "Filled nulls with 'unknown' and added a binary device_id_missing flag so the model can learn 'missing device' as a signal."

  **"is_new_device_strategy":** "Nulls filled with -1 (unknown category) and a separate is_new_device_missing flag added."

  **"merchant_name_strategy":** "Nulls filled with 'unknown'. For Payroll and P2P transfers nulls are expected (no merchant); for other categories only a small number of genuine gaps."

  **"negative_amounts":** "291 negative amounts (refunds/credits) kept as legitimate; added an is_refund flag.",
  **"outliers":** "Extreme amounts retained (financial long-tail). Log transform applied later in feature engineering, not here."

  **"is_cross_border_check":** "Confirmed 100% consistent with transaction_country vs home_country; no changes."

  **"accounts_cleaning":* "No nulls or duplicates. account_created_date parsed to datetime; whitespace trimmed."

  **"merge":** "Left join tx <- acc on account_id. Duplicate columns (account_type, home_country, currency, kyc_tier) dropped from acc side before merge to avoid _x/_y pairs. Referential integrity confirmed: every account_id in tx exists in acc."

  **"merged_rows":** 126000

  **"merged_cols":** 30

  **"fraud_rate_pct":** 2.66

### 10. Summary

**What we did.**
- Loaded `tx` (126,000 × 24) and `acc` (7,200 × 8); no duplicates, referential integrity OK.
- Cleaned `tx`: parsed timestamps, confirmed `day_of_week`/`hour_of_day`, validated
  categoricals, filled `device_id`/`is_new_device` nulls with "unknown" + missing flags,
  filled `merchant_name` nulls with "unknown", kept negative amounts with an `is_refund`
  flag, confirmed `is_cross_border` consistency.
- Cleaned `acc`: parsed `account_created_date`, trimmed whitespace; table was already clean.
- Merged into `df` via left join on `account_id` (dropping shared columns from the acc side).
- Saved `processed_data/test_cleaned_data.csv` and `cleaning_log.json`.

**Key numbers to remember for later notebooks.**
- Fraud rate: **~2.66%** (heavily imbalanced → stratified split + class weighting).
- New columns added: `device_id_missing`, `is_new_device_missing`, `is_refund`.
- `status` has a strong fraud signal (Reversed = 33% fraud) but is known only
  *after* the transaction → it will be **excluded** from real-time scoring features
  (documented in feature engineering as a target-leakage guard).

